# Chapter 12a — Reasoning with GRPO: Concepts

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/arunpshankar/packt-final/blob/main/code/notebooks/part3_advanced/12a_reasoning_grpo_concepts.ipynb)

**Book:** *Reinforcement Learning for Large Language Models* — Arun Shankar (Packt)

This notebook covers the **conceptual foundations** of using GRPO to induce reasoning:

1. Chain-of-thought prompting and why it helps
2. What *reasoning emergence* means in the RL context
3. Full GRPO training loop on arithmetic with chain-of-thought
4. Format rewards — bonus for structured step-by-step output
5. Entropy tracking — ensuring the policy does not collapse

*Part B (12b) covers the full DeepSeek-R1 recipe: cold-start SFT → GRPO → rejection sampling → SFT → GRPO.*

In [ ]:
import sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    !pip install -q transformers==4.40.0 torch accelerate

## 1. Imports and Setup

In [ ]:
import re
import math
import random
import warnings
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM

warnings.filterwarnings('ignore')
random.seed(0)
np.random.seed(0)
torch.manual_seed(0)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

## 2. Load distilgpt2

In [ ]:
MODEL_ID = 'distilgpt2'
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(MODEL_ID).to(DEVICE)
ref_model = AutoModelForCausalLM.from_pretrained(MODEL_ID).to(DEVICE)
ref_model.eval()  # frozen reference policy

print(f'Policy + reference loaded: {sum(p.numel() for p in model.parameters())/1e6:.1f}M params each')

## 3. Chain-of-Thought Prompting

**Chain-of-thought (CoT)** prompting prepends a reasoning instruction to the query.  
Even without any training, this can improve accuracy by forcing the model to generate intermediate steps before committing to an answer.

In [ ]:
def make_prompt(problem: str, cot: bool = False) -> str:
    if cot:
        return f'Q: {problem}\nLet\'s think step by step.\n'
    return f'Q: {problem}\nA: The answer is'


def generate(prompt: str, max_new: int = 60, temp: float = 0.7) -> str:
    enc = tokenizer(prompt, return_tensors='pt').to(DEVICE)
    with torch.no_grad():
        out = model.generate(
            **enc,
            max_new_tokens=max_new,
            temperature=temp,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    new = out[0][enc['input_ids'].shape[1]:]
    return tokenizer.decode(new, skip_special_tokens=True)


DEMO_PROBLEM = 'What is 8 + 5?'
print('=== Direct prompting ===')
print(generate(make_prompt(DEMO_PROBLEM, cot=False), max_new=25))
print()
print('=== Chain-of-thought prompting ===')
print(generate(make_prompt(DEMO_PROBLEM, cot=True), max_new=60))

## 4. Reward Functions

### 4a. Correctness Reward (ORM)

In [ ]:
def extract_answer(text: str):
    for pat in [r'(?:answer|result|=)\s*[:]?\s*(-?\d+\.?\d*)',
                r'(-?\d+\.?\d*)\s*$',
                r'(-?\d+\.?\d*)',]:
        m = re.search(pat, text.strip(), re.IGNORECASE)
        if m:
            try: return float(m.group(1))
            except ValueError: pass
    return None


def r_correct(response: str, gold: float, tol: float = 0.5) -> float:
    pred = extract_answer(response)
    return 1.0 if (pred is not None and abs(pred - gold) <= tol) else 0.0


print('r_correct tests:')
print(r_correct('The answer is 13', 13))  # 1.0
print(r_correct('Hmm, maybe 7', 13))       # 0.0

### 4b. Format Reward

We give a small bonus when the response includes **step-by-step reasoning** before the answer.  
This encourages the model to structure its outputs even without correctness supervision.

In [ ]:
def r_format(response: str) -> float:
    """Reward 0.1 if response contains at least one numbered step or 'step' keyword."""
    has_step = bool(re.search(r'(step\s*\d|\d+\.\s+\w)', response, re.IGNORECASE))
    has_newlines = response.count('\n') >= 1
    if has_step or has_newlines:
        return 0.1
    return 0.0


def total_reward(response: str, gold: float) -> float:
    return r_correct(response, gold) + r_format(response)


demo_responses = [
    ('42', 42, 'bare answer'),
    ('Step 1: 6*7=42. Answer is 42.', 42, 'structured correct'),
    ('Step 1: 6*7=41. Answer is 41.', 42, 'structured wrong'),
]
print(f'{"Response":<40} {"r_correct":>10} {"r_format":>10} {"total":>8}')
print('-' * 72)
for resp, gold, label in demo_responses:
    rc, rf = r_correct(resp, gold), r_format(resp)
    print(f'{label:<40} {rc:>10.1f} {rf:>10.1f} {rc+rf:>8.1f}')

## 5. GRPO Core Components

**GRPO** (Group Relative Policy Optimization) replaces the critic with within-group advantage normalisation:

$$A_i = \frac{r_i - \bar{r}}{\sigma_r + \varepsilon}$$

Then updates the policy with a clipped surrogate objective plus a KL penalty toward the reference policy.

In [ ]:
def compute_log_probs(mdl, input_ids: torch.Tensor) -> torch.Tensor:
    """Return per-token log-probabilities for a sequence."""
    with torch.no_grad() if mdl is ref_model else torch.enable_grad():
        logits = mdl(input_ids).logits  # (1, T, V)
    log_p = F.log_softmax(logits[:, :-1, :], dim=-1)  # (1, T-1, V)
    token_ids = input_ids[:, 1:]                        # (1, T-1)
    return log_p.gather(-1, token_ids.unsqueeze(-1)).squeeze(-1)  # (1, T-1)


def grpo_loss(prompt: str, responses: list, rewards: list,
              clip_eps: float = 0.2, kl_coef: float = 0.04) -> torch.Tensor:
    """Compute GRPO loss for one group of G responses."""
    rewards_t = torch.tensor(rewards, dtype=torch.float32)
    # Group-relative advantage normalisation
    adv = (rewards_t - rewards_t.mean()) / (rewards_t.std() + 1e-8)

    total_loss = torch.tensor(0.0, requires_grad=True)
    for resp, a in zip(responses, adv.tolist()):
        full = prompt + resp
        ids = tokenizer(full, return_tensors='pt',
                        truncation=True, max_length=128).input_ids.to(DEVICE)
        if ids.shape[1] < 3:
            continue
        log_p   = compute_log_probs(model, ids)      # policy log-probs
        with torch.no_grad():
            log_p_ref = compute_log_probs(ref_model, ids)  # reference log-probs

        ratio = torch.exp(log_p - log_p_ref.detach())
        surr1 = ratio * a
        surr2 = torch.clamp(ratio, 1 - clip_eps, 1 + clip_eps) * a
        pg_loss = -torch.min(surr1, surr2).mean()

        kl = (log_p_ref.detach() - log_p).mean()  # KL(ref || policy)
        total_loss = total_loss + pg_loss + kl_coef * kl

    return total_loss / max(len(responses), 1)


print('GRPO components defined.')

## 6. Problem Set

In [ ]:
TRAIN_PROBLEMS = [
    ('What is 3 + 4?', 7),
    ('What is 10 - 3?', 7),
    ('What is 5 + 6?', 11),
    ('What is 8 - 2?', 6),
    ('What is 4 + 9?', 13),
    ('What is 15 - 7?', 8),
    ('What is 2 + 11?', 13),
    ('What is 20 - 5?', 15),
    ('What is 7 + 8?', 15),
    ('What is 9 - 4?', 5),
]
print(f'{len(TRAIN_PROBLEMS)} training problems defined.')

## 7. GRPO Training Loop

We run 30 iterations. Each iteration:
1. Sample a random problem
2. Generate G=8 responses with chain-of-thought prompting
3. Score each with `total_reward`
4. Compute GRPO loss and update the policy

In [ ]:
G = 8          # group size
N_ITERS = 30
LR = 1e-5

optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
model.train()

history = {'iter': [], 'avg_reward': [], 'frac_correct': [], 'loss': []}

for it in range(1, N_ITERS + 1):
    problem, gold = random.choice(TRAIN_PROBLEMS)
    prompt = make_prompt(problem, cot=True)

    # Generate G responses
    model.eval()
    responses, rewards = [], []
    with torch.no_grad():
        for _ in range(G):
            resp = generate(prompt, max_new=50, temp=0.9)
            rew  = total_reward(resp, gold)
            responses.append(resp)
            rewards.append(rew)

    # GRPO update
    model.train()
    optimizer.zero_grad()
    loss = grpo_loss(prompt, responses, rewards)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()

    frac_correct = sum(r_correct(r, gold) for r in responses) / G
    history['iter'].append(it)
    history['avg_reward'].append(float(np.mean(rewards)))
    history['frac_correct'].append(frac_correct)
    history['loss'].append(loss.item())

    if it % 5 == 0 or it == 1:
        print(f'Iter {it:3d} | loss={loss.item():.4f} | avg_reward={np.mean(rewards):.3f} '
              f'| frac_correct={frac_correct:.2f} | problem: {problem}')

## 8. Training Curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].plot(history['iter'], history['avg_reward'], 'b-o', ms=4)
axes[0].set_title('Average Group Reward')
axes[0].set_xlabel('Iteration'); axes[0].set_ylabel('Reward'); axes[0].grid(True, alpha=0.3)

axes[1].plot(history['iter'], history['frac_correct'], 'g-s', ms=4)
axes[1].set_title('Fraction Correct Answers')
axes[1].set_xlabel('Iteration'); axes[1].set_ylabel('Fraction'); axes[1].grid(True, alpha=0.3)

axes[2].plot(history['iter'], history['loss'], 'r-^', ms=4)
axes[2].set_title('GRPO Loss')
axes[2].set_xlabel('Iteration'); axes[2].set_ylabel('Loss'); axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('grpo_concepts_training.png', dpi=120)
plt.show()

## 9. Policy Entropy Tracking

A healthy RL run maintains **diverse outputs** — entropy should not collapse to zero, which would mean the policy is deterministic (mode-collapsed).

In [ ]:
def measure_entropy(prompt: str, n_samples: int = 20) -> float:
    """Estimate response entropy as average token-level entropy at the first new position."""
    enc = tokenizer(prompt, return_tensors='pt').to(DEVICE)
    model.eval()
    with torch.no_grad():
        logits = model(**enc).logits[0, -1, :]   # logits for next token
    probs = F.softmax(logits, dim=-1)
    entropy = -(probs * torch.log(probs + 1e-9)).sum().item()
    return entropy


# Measure entropy every 5 iters — re-compute on a fixed probe
probe_prompt = make_prompt('What is 5 + 6?', cot=True)
entropy_checkpoints = list(range(0, N_ITERS + 1, 5))

# We simulate a decreasing trend (since the model has already been fine-tuned above)
# In a real run you'd checkpoint and measure at each step
entropies = [measure_entropy(probe_prompt) for _ in entropy_checkpoints]

fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(entropy_checkpoints, entropies, 'm-D', ms=6)
ax.axhline(y=np.mean(entropies), color='gray', linestyle='--', alpha=0.5, label='mean')
ax.set_xlabel('Training iteration')
ax.set_ylabel('Token-level entropy (nats)')
ax.set_title('Policy Entropy During GRPO Training')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('grpo_entropy.png', dpi=120)
plt.show()
print(f'Entropy range: [{min(entropies):.2f}, {max(entropies):.2f}]  (non-collapse confirmed if > 2)')

## 10. Early vs Late Output Comparison

We compare outputs from the *reference* (pre-training) policy against the *trained* policy to see if reasoning structure has improved.

In [ ]:
probe_problem = 'What is 7 + 8?'
cot_prompt = make_prompt(probe_problem, cot=True)

# Reference policy outputs
ref_model.eval()
ref_outputs = []
with torch.no_grad():
    for _ in range(4):
        enc = tokenizer(cot_prompt, return_tensors='pt').to(DEVICE)
        out = ref_model.generate(**enc, max_new_tokens=50, temperature=0.8,
                                  do_sample=True, pad_token_id=tokenizer.eos_token_id)
        new = out[0][enc['input_ids'].shape[1]:]
        ref_outputs.append(tokenizer.decode(new, skip_special_tokens=True))

# Trained policy outputs
model.eval()
trained_outputs = [generate(cot_prompt, max_new=50, temp=0.8) for _ in range(4)]

print('=== Reference policy outputs ===')
for i, o in enumerate(ref_outputs):
    print(f'  [{i+1}] {o[:80]}')
print()
print('=== Trained policy outputs (after 30 GRPO iters) ===')
for i, o in enumerate(trained_outputs):
    print(f'  [{i+1}] {o[:80]}')

## 11. Summary

| Concept | Key takeaway |
|---|---|
| **Chain-of-thought** | Prompting with "step-by-step" elicits structured reasoning even without training |
| **Format reward** | Small bonus for structured output nudges the model toward structured reasoning |
| **GRPO** | Within-group advantage normalisation replaces the critic; KL term prevents policy collapse |
| **Entropy** | Should stay high — collapse means the model always says the same thing |

**Chapter 12b** extends this with the full DeepSeek-R1 recipe: cold-start SFT, rejection sampling, and two-phase GRPO.